# Examples for paper **"Effective computation of centralizers of ODOs"**

In this document we provide the code to reproduce the computations perfomed in the Examples of the aforementioned paper that can be found in ((add link)).

In [2]:
import sys
sys.path.insert(0, "..") # dalgebra is here

from dalgebra import *
from dalgebra.commutators import *

import logging
logging.getLogger("dalgebra").setLevel(logging.INFO)

def dict_to_point(point: dict, variables: list):
    return tuple(point[str(v)] for v in variables)

## Example 4.2 (Boussinesq at level $m=4$)

In this example we explorer results from previous articles to show the GD hierarchy for a generic operator of order $3$ (i.e., the Boussinesq case) at level $m=4$.

In [3]:
# Getting a generic operator
L = generic_normal(3); z = L.parent().gen("z")
L

u_2_0*z_1 + u_3_0*z_0 + z_3

In [4]:
# Computing the almost commuting basis up to level 4
P,H= zip(*[almost_commuting_wilson(3,i) for i in range(6)])

In [5]:
for (i,p) in enumerate(P[:5]):
    print(i, "-->", p)

0 --> z_0
1 --> z_1
2 --> 2/3*u_2_0*z_0 + z_2
3 --> u_2_0*z_1 + u_3_0*z_0 + z_3
4 --> 4/3*u_2_0*z_2 + 2/9*u_2_0^2*z_0 + 2/3*u_2_1*z_1 + 2/9*u_2_2*z_0 + 4/3*u_3_0*z_1 + 2/3*u_3_1*z_0 + z_4


In [6]:
H[1]

(-u_3_1, -u_2_1)

In [7]:
H[2]

(2/3*u_2_0*u_2_1 + 2/3*u_2_3 - u_3_2, u_2_2 - 2*u_3_1)

In [8]:
H[4]

(2/3*u_2_0*u_2_3 - 2/3*u_2_0*u_3_2 + 4/9*u_2_0^2*u_2_1 + 4/3*u_2_1*u_2_2 - 2/3*u_2_1*u_3_1 + 2/9*u_2_5 - 4/3*u_3_0*u_3_1 - 1/3*u_3_4,
 2/3*u_2_0*u_2_2 - 4/3*u_2_0*u_3_1 - 4/3*u_2_1*u_3_0 + 2/3*u_2_1^2 + 1/3*u_2_4 - 2/3*u_3_3)

## Example 5.2 (Specialization map for $n=3$ and $m=4$)

In this example we specialized the hierarchy system from Example 3.2 to two specific cases where we evaluate the coefficient functions $u_2,u_3$ to certain types of functions

### Example 5.2.1: rational coefficients

In [9]:
B.<x> = PolynomialRing(QQ)
DB = DifferentialRing(B, [1]) # this creates (Q[x], dx)
x = DB("x")

Us = {"u_2": -6/x^2, "u_3": 12/x^3}

In [10]:
H_eval_rational = [[coeff(**Us) for coeff in H[i]] for i in Jset(4,3,0)]
H_eval_rational

[[36/x^4, ((-12)/x^3)], [((-96)/x^5), 36/x^4], [0, 0]]

In [11]:
Matrix(H_eval_rational).transpose() # linear system matrix of the hierarchy

[     36/x^4 ((-96)/x^5)           0]
[((-12)/x^3)      36/x^4           0]

### Example 5.2.2: hyperbolic coefficients

In [12]:
B = QQ
DB = DifferentialRing(QQ, 0) # this is (Q,0)
E.<cosh> = DElliptic(DB, "cosh_p^2 - cosh^2 + 1")

Us = {"u_2": 6/cosh^2, "u_3": 0}

In [13]:
H_eval_hyperbolic = [[coeff(**Us) for coeff in H[i]] for i in Jset(4,3,0)]
H_eval_hyperbolic

[[0, (12/cosh^3*cosh_p)],
 [(((-32*cosh^2 + 48)/cosh^5)*cosh_p), ((24*cosh^2 - 36)/cosh^4)],
 [(((-128/3*cosh^2 + 64)/cosh^5)*cosh_p), ((32*cosh^2 - 48)/cosh^4)]]

In [14]:
Matrix(H_eval_hyperbolic).transpose() # linear system matrix of the hierarchy

[                                     0    (((-32*cosh^2 + 48)/cosh^5)*cosh_p) (((-128/3*cosh^2 + 64)/cosh^5)*cosh_p)]
[                    (12/cosh^3*cosh_p)              ((24*cosh^2 - 36)/cosh^4)              ((32*cosh^2 - 48)/cosh^4)]

## Example 6.6 (Continuation of 5.2.2)

This examples shows how Algorithm 4 computes the filtered basis in the hyperbolic case. So, recalling from the previous example, we have the differential operator 
$$\mathbb{L} = \partial^3 + \frac{6}{\cosh(x)^2}\partial.$$

In Example 5.2.2, we have computed the induced linear system by the homogeneous hierarchy:
$$\begin{pmatrix}
0 & \displaystyle\frac{48-32\cosh(x)^2}{\cosh(x)^5}\sinh(x) & \displaystyle\frac{192-128\cosh(x)^2}{3\cosh(x)^5}\sinh(x)\\
\displaystyle\frac{12 \sinh(x)}{\cosh(x)^3} & \displaystyle\frac{24\cosh(x)^2 - 36}{\cosh(x)^4} & \displaystyle\frac{32\cosh(x)^2 - 48}{\cosh(x)^4}
\end{pmatrix}$$

From this system, we apply Algorithm 3 to extend the system to a new linear system in constant coefficients whose solutions are the constant solutions of the previous homogeneous linear system.

Algorithm 3 works by taking linearly independent terms (i.e., taking monomials in numerators of each row) and collecting all the coefficients on the same row. This is done in the code with the following piece of code:

In [15]:
## Same as in Example 5.2.2
B = QQ
DB = DifferentialRing(QQ, 0) # this is (Q,0)
E.<cosh> = DElliptic(DB, "cosh_p^2 - cosh^2 + 1")

Us = {"u_2": 6/cosh^2, "u_3": 0}
H_eval_hyperbolic = Matrix([[coeff(**Us) for coeff in H[i]] for i in Jset(4,3,0)]).transpose()

In [16]:
system = E.system_for_constant_solutions([list(row) for row in H_eval_hyperbolic])[0]
system

/home/jimenezp/sage/sage/src/sage/modules/free_module.py:272: UserWarning: You are constructing a free module
over a noncommutative ring. Sage does not have a concept
of left/right and both sided modules, so be careful.
It's also not guaranteed that all multiplications are
done from the right side.
  warn("You are constructing a free module\n"
/home/jimenezp/sage/sage/src/sage/modules/free_module.py:2004: UserWarning: You are constructing a free module
over a noncommutative ring. Sage does not have a concept
of left/right and both sided modules, so be careful.
It's also not guaranteed that all multiplications are
done from the right side.
  warn("You are constructing a free module\n"


[     0     48     64]
[     0    -32 -128/3]
[    12      0      0]
[     0    -36    -48]
[     0     24     32]

Finding a solution to this linear system provides a flag of constants such that the corresponding linear combination of Wilson's almost commuting basis commutes with the operator $\mathbb{L}$:

In [17]:
system.right_kernel()

Vector space of degree 3 and dimension 1 over Rational Field
Basis matrix:
[   0    1 -3/4]

Hence, any operator of the form $c(P_2(\mathbb{L}) -\frac{3}{4}P_4(\mathbb{L}))$ commutes with $\mathbb{L}$. In particular, there is a **unique** monic differential operator of order $4$ that commutes with $\mathbb{L}$:
$$Z_4 = -\frac{4}{3}(P_2(\mathbb{L}) -\frac{3}{4}P_4(\mathbb{L})) = $$

In [18]:
Z_4 = (-4/3)*(P[2](**Us) + (-3/4)*P[4](**Us)); Z_4

-(8/cosh^3*cosh_p)*z_1 + ((-4/3*cosh^2 + 8)/cosh^2)*z_2 + z_4

In [19]:
Z_4.lie_bracket(L(**Us), "z") ## They commute

0

This shows that the level of $\mathbb{L}$ is exactly $4$. We can then apply Algorithm 4 to compute the full centralizer as a $\mathbf{C}[\mathbb{L}]$-module by increasing the order we are looking on the centralizer and solving the same system:

#### Case with $m=5$

Since we have $\mathbb{L}$ of order $3$ and level $4$, when we do the iteration of Algorithm $4$ with $m=5$, we now consider the system with $J = \{1,2,5\}$ (we did not add $4$ since we found an element in the centralizer for such order). Then we perform similar computations:

In [20]:
Jset(5,3,0,4)

[1, 2, 5]

In [21]:
H_eval_hyperbolic_5 = Matrix([[coeff(**Us) for coeff in H[i]] for i in Jset(5,3,0,4)]).transpose()
H_eval_hyperbolic_5

[                                  0 (((-32*cosh^2 + 48)/cosh^5)*cosh_p)                                   0]
[                 (12/cosh^3*cosh_p)           ((24*cosh^2 - 36)/cosh^4)               -(64/3/cosh^3*cosh_p)]

In [22]:
system_5 = E.system_for_constant_solutions([list(row) for row in H_eval_hyperbolic_5])[0]
system_5

[    0    48     0]
[    0   -32     0]
[   12     0 -64/3]
[    0   -36     0]
[    0    24     0]

In [23]:
system_5.right_kernel()

Vector space of degree 3 and dimension 1 over Rational Field
Basis matrix:
[   1    0 9/16]

In [24]:
Z_5 = P[5](**Us) + (16/9)*P[1](**Us)
print(Z_5)
Z_5.lie_bracket(L(**Us), "z")

((16/9*cosh^4 + 80/3*cosh^2 - 20)/cosh^4)*z_1 - (20/cosh^3*cosh_p)*z_2 + 10/cosh^2*z_3 + z_5


0

This concludes the exection of Algorithm 4. This algorithm has the name of `GetCentralizer` in the module and return, precisely, this pair of operators:

In [25]:
%%time
GetCentralizer({0: Us["u_3"], 1: Us["u_2"]}, 4, starting_level=4, ignore_bound=True) 
## The output is L, the Goodearl's basis and the flag of constants for the first element in the centralizer.

CPU times: user 59.8 ms, sys: 2.96 ms, total: 62.8 ms
Wall time: 62.6 ms


(6/cosh^2*z_1 + z_3,
 [z_0,
  -(8/cosh^3*cosh_p)*z_1 + ((-4/3*cosh^2 + 8)/cosh^2)*z_2 + z_4,
  ((16/9*cosh^4 + 80/3*cosh^2 - 20)/cosh^4)*z_1 - (20/cosh^3*cosh_p)*z_2 + 10/cosh^2*z_3 + z_5],
 [0, -4/3, 1])

## Example 6.7 ([Pogorelov and Zheglov, Example 22])

In the paper by Pogorelov and Zheglov (see https://doi.org/10.1134/S1995080217060117), they presented several pairs of differential operators with rational coefficients that commute. Two of them (see Example 22 on the paper) have order $4$ and $6$. Let us use Algorithm 4 to compute the centralizer of the operator of order $4$ as a $\mathbf{C}[\mathbb{L}]$-module.

In [26]:
B.<x> = PolynomialRing(QQ)
DB = DifferentialRing(B, [1]) # this creates (Q[x], dx)
x = DB("x")
y = x^5 + 30 # base polynomial in denominator

Us = {"u_2": -20*x^3*(x^5-120)/y^2, "u_3": -3000*x^2*(7*x^5-90)/y^3, "u_4": 18000*x*(3*x^10-145*x^5 + 450)/y^4}

In [27]:
%%time
L_6_8, GB, _ = GetCentralizer({0:Us["u_4"], 1: Us["u_3"], 2: Us["u_2"]}, 12, starting_level=6, ignore_bound=True)

CPU times: user 461 ms, sys: 29.6 ms, total: 490 ms
Wall time: 493 ms


In [28]:
# The original operator of order 4
L_6_8

((54000*x^11 - 2610000*x^6 + 8100000*x)/(x^20 + 120*x^15 + 5400*x^10 + 108000*x^5 + 810000))*z_0 + ((-21000*x^7 + 270000*x^2)/(x^15 + 90*x^10 + 2700*x^5 + 27000))*z_1 + ((-20*x^8 + 2400*x^3)/(x^10 + 60*x^5 + 900))*z_2 + z_4

In [29]:
# The operator of order 6 in the paper
GB[1]

((4410000*x^19 - 946350000*x^14 + 25474500000*x^9 - 87480000000*x^4)/(x^30 + 180*x^25 + 13500*x^20 + 540000*x^15 + 12150000*x^10 + 145800000*x^5 + 729000000))*z_0 + ((-2304000*x^15 + 247590000*x^10 - 2616300000*x^5 + 972000000)/(x^25 + 150*x^20 + 9000*x^15 + 270000*x^10 + 4050000*x^5 + 24300000))*z_1 + ((531000*x^11 - 22815000*x^6 + 52650000*x)/(x^20 + 120*x^15 + 5400*x^10 + 108000*x^5 + 810000))*z_2 + ((60*x^12 - 63900*x^7 + 729000*x^2)/(x^15 + 90*x^10 + 2700*x^5 + 27000))*z_3 + ((-30*x^8 + 3600*x^3)/(x^10 + 60*x^5 + 900))*z_4 + z_6

In this case, we knew the level of the operator was $6$ (since in the paper they have computed a non-trivial element in the centralizer for such order). Although the theory could not guarantee we will find further elements for Goodearl's basis, in this particular case we were lucky enough showing that we have elements of order $7$ and $9$:

In [30]:
# The operator of order 7 that commutes with L_6_8
GB[2]

((-46777500*x^23 + 17255700000*x^18 - 905357250000*x^13 + 8853705000000*x^8 - 10461150000000*x^3)/(x^35 + 210*x^30 + 18900*x^25 + 945000*x^20 + 28350000*x^15 + 510300000*x^10 + 5103000000*x^5 + 21870000000))*z_0 + ((26617500*x^19 - 5566050000*x^14 + 146994750000*x^9 - 493290000000*x^4)/(x^30 + 180*x^25 + 13500*x^20 + 540000*x^15 + 12150000*x^10 + 145800000*x^5 + 729000000))*z_1 + ((-6914250*x^15 + 721980000*x^10 - 7470225000*x^5 + 2551500000)/(x^25 + 150*x^20 + 9000*x^15 + 270000*x^10 + 4050000*x^5 + 24300000))*z_2 + ((-105*x^16 + 1045800*x^11 - 44068500*x^6 + 96390000*x)/(x^20 + 120*x^15 + 5400*x^10 + 108000*x^5 + 810000))*z_3 + ((105*x^12 - 93450*x^7 + 1039500*x^2)/(x^15 + 90*x^10 + 2700*x^5 + 27000))*z_4 + ((-35*x^8 + 4200*x^3)/(x^10 + 60*x^5 + 900))*z_5 + z_7

In [31]:
# The operator of order 9 that commutes with L_6_8
GB[3]

((-6891885000*x^31 + 6103528200000*x^26 - 858171510000000*x^21 + 30302634600000000*x^16 - 283026777750000000*x^11 + 537818292000000000*x^6 - 56687040000000000*x)/(x^45 + 270*x^40 + 32400*x^35 + 2268000*x^30 + 102060000*x^25 + 3061800000*x^20 + 61236000000*x^15 + 787320000000*x^10 + 5904900000000*x^5 + 19683000000000))*z_0 + ((4403565000*x^27 - 2542266000000*x^22 + 226011870000000*x^17 - 4531464000000000*x^12 + 18762819750000000*x^7 - 7203978000000000*x^2)/(x^40 + 240*x^35 + 25200*x^30 + 1512000*x^25 + 56700000*x^20 + 1360800000*x^15 + 20412000000*x^10 + 174960000000*x^5 + 656100000000))*z_1 + ((-1331842500*x^23 + 473193900000*x^18 - 24350787000000*x^13 + 233965260000000*x^8 - 266868675000000*x^3)/(x^35 + 210*x^30 + 18900*x^25 + 945000*x^20 + 28350000*x^15 + 510300000*x^10 + 5103000000*x^5 + 21870000000))*z_2 + ((-1350*x^24 + 250263000*x^19 - 50470290000*x^14 + 1307801700000*x^9 - 4265743500000*x^4)/(x^30 + 180*x^25 + 13500*x^20 + 540000*x^15 + 12150000*x^10 + 145800000*x^5 + 729000000)

## Example 7.5 (ODOs of level $M=4$ and hyperbolic coefficients)

In this example we explore the computation of the level variety for an operator of order $3$ with hyperbolic coefficients at level $4$. These computations show what is done inside the method `GetEquationsForLevel`, which already include the removal of previous level varieties.

This example begins with a generic operator of order $3$ with hyperbolic coefficients, namely:
$$\mathbb{L} = \partial^3 + \frac{a_2}{\cosh(x)^2}\partial + \frac{a_3\sinh(x)}{\cosh(x)^3},$$
for two constants $a_2,a_3$.

We now consider the hierarchy equations (similar to example 5.2, but keeping the generic coefficients) after specializing the coefficients. This lead to the linear system with constant coefficients whose solutions are the exact constant solutions to the hierarchy equations.

In [32]:
## Setting up the algebraic setting in `dalgebra`.
B = DElliptic(DifferentialRing(QQ), "cosh^2 - cosh_p^2  - 1", "cosh")

DO = DifferentialPolynomialRing(B, "z")
DO_wa = DO.add_constants("a_2", "a_3")
B_wa = DO_wa.base()

cosh = B_wa.gen(); sinh = B_wa.gen_p()
a = [None,None] + [B_wa(f"a_{i}") for i in range(2,4)]
Us = {"u_2": a[2]/cosh^2, "u_3": a[3]*sinh/cosh^3}

2025-04-30 12:56:24 INFO     No operation is given: we set a zero derivative.


In [33]:
H4_eval_hyperbolic_gen = Matrix([[coeff(**Us) for coeff in H[i]] for i in Jset(4,3,0)]).transpose()

In [34]:
system_4 = B_wa.system_for_constant_solutions([list(row) for row in H4_eval_hyperbolic_gen])[0]
system_4

[                                                          -3*a_3                                                                0                                                                0]
[                                                           2*a_3                                                                0                                                                0]
[                                                               0                                     -4/3*a_2^2 + 16*a_2 + 12*a_3        -16*a_2^2 - 16/3*a_2*a_3 + 8/3*a_3^2 + 320/3*a_2 + 80*a_3]
[                                                               0                                                -16/3*a_2 - 4*a_3                                             -64/9*a_2 - 16/3*a_3]
[                                                               0                                                                0 -8/9*a_2^3 + 32*a_2^2 + 12*a_2*a_3 - 4*a_3^2 - 160*a_2 - 120*a_3]
[              

Now that we have the system ready, we know that the level variety is the variety of the ideal generated by the minors of the previous system. We can analyze this ideal as follows:

In [35]:
minors_4 = [el.numerator() for el in system_4.minors(3) if el != 0]
gb_4 = ideal(minors_4).groebner_basis()
gb_4

[a_2^4 - 6*a_2^3,
 a_3^4 + 12*a_3^3,
 a_2^2*a_3 - 1/4*a_3^3,
 a_2*a_3^2 + 1/2*a_3^3]

In [36]:
# Computing the variety for this ideal
X_4 = [dict_to_point(point,a[2:]) for point in ideal(gb_4).radical().variety()]
X_4

[(0, 0), (6, 0), (6, -12)]

To remove the previous level varieties, we need to compute the points and remove the repeated cases:

In [37]:
## Computing the level variety at level 2
H2_eval_hyperbolic_gen = Matrix([[coeff(**Us) for coeff in H[i]] for i in Jset(2,3,0)]).transpose()
system_2 = B_wa.system_for_constant_solutions([list(row) for row in H2_eval_hyperbolic_gen])[0]
minors_2 = [el.numerator() for el in system_2.minors(2) if el != 0]
gb_2 = ideal(minors_2).groebner_basis()
X_2 = [dict_to_point(point,a[2:]) for point in ideal(gb_2).radical().variety()]

In [38]:
gb_2

[a_2^2, a_2*a_3, a_3^2]

In [39]:
## Removing points from level two at level 4
[el for el in X_4 if not el in X_2]

[(6, 0), (6, -12)]

## Example 7.6 (Level varieties for rational coefficients with $n=4$)

Similar to the previous example, we are going to study now the level varieties of a generic operator of order $4$ with rational coefficients. Namely, we consider the generic differential operator
$$\mathbb{L} = \partial^4 + \frac{a_2}{x^2}\partial^2 + \frac{a_3}{x^3}\partial + \frac{a_4}{x^4},$$
for arbitrary constants $a_2,a_3,a_4$.

We consider the hierarchy equations, we extend to the system for constant solutions and see the ideal whose variety is the level variety:

In [40]:
B.<x> = PolynomialRing(QQ)
DB = DifferentialRing(B, [1]) # this creates (Q[x], dx)

DO = DifferentialPolynomialRing(B.fraction_field(), "z")
DO_wa = DO.add_constants("a_2", "a_3", "a_4")
B_wa = DO_wa.base()

x = B_wa.gens()[0]
a = (None,None) + B_wa.gens()[1:]
Us = {"u_2": a[2]/x^2, "u_3": a[3]/x^3, "u_4": a[4]/x^4}

We start by computing the hierarchy for $n=4$ and $m\leq 6$:

In [41]:
%%time
# Computing the almost commuting basis up to level 4
P4,H4= zip(*[almost_commuting_wilson(4,i) for i in range(7)])

CPU times: user 11 ms, sys: 242 µs, total: 11.2 ms
Wall time: 11 ms


In [42]:
H4_6_rational_gen = Matrix([[coeff(**Us) for coeff in H4[i]] for i in Jset(6,4,0)]).transpose()
system_4_6 = B_wa.system_for_constant_solutions([list(row) for row in H4_6_rational_gen])[0]
show(system_4_6)

[                                                                                                                                                                                                                          4*a_4                                                                                                                                                                                                                               0                                                                                                                                                                                                                               0                                                                                                                                                                                                                               0                                                                                                                                                                                                                               0]
[                                                                                                                                                                                                                              0                                                                                                                                                                                             3*a_2^2 - a_2*a_3 + 60*a_2 - 20*a_4                                                                                                                                                                                                                               0                                                                                                                                                                                                                               0                                                                                                                                                                                                                               0]
[                                                                                                                                                                                                                              0                                                                                                                                                                                                                               0                                                                                                                                                    9*a_2^2 + 27/4*a_2*a_3 - 9/4*a_3^2 + 3*a_2*a_4 + 270*a_2 + 270*a_3 + 120*a_4                                                                                                                                                                                                                               0                                                                                                                                                                                                                               0]
[                                                                                                                                                                                                                              0                                                                                                                                                                                                                               0                                                                                                                                                                                          

From this linear system, we take all minors of size $5\times 5$, and compute the resulting ideal (which defines the level variety $\mathcal{X}_6$):

In [43]:
%%time
I_6 = ideal([el for el in system_4_6.minors(5) if el != 0])
PD_I_6 = [el.radical() for el in I_6.primary_decomposition()]
for component in PD_I_6:
    component = component.radical()
    if component.dimension() == 0:
        print(component.dimension(), " -> ", [dict_to_point(point,a[2:]) for point in component.variety()], " -> ", component)
    else:
        print(component.dimension(), " |-> ", component)

1  |->  Ideal (4*a_2*a_3 + a_3^2 + 24*a_3 + 16*a_4, a_2^2 + 32*a_2 - 6*a_3 - 4*a_4 + 240) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-24, 24, 0)]  ->  Ideal (a_4, a_3 - 24, a_2 + 24) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-24, 72, -72)]  ->  Ideal (a_4 + 72, a_3 - 72, a_2 + 24) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-36, 24, 144)]  ->  Ideal (a_4 - 144, a_3 - 24, a_2 + 36) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-36, 120, 0)]  ->  Ideal (a_4, a_3 - 120, a_2 + 36) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
1  |->  Ideal (2*a_2 + a_3, a_3^2 - 152*a_3 - 16*a_4 + 4480) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
1  |->  Ideal (2*a_2 + a_3, a_3^2 - 24*a_3 - 16*a_4) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-60, 120, 216)]  ->  Ideal (a_4 - 216, a_3 - 120, a

This variety can decompose into these irreducible components. We can see that there are three components of dimension 1 and 15 of dimension 0. We can remove those that come from previous level varieties. In order to do so, we are going to compute all level varieties and analyze them individually.

#### Level variety at level $m=1$

In [44]:
%%time
H4_1_rational_gen = Matrix([[coeff(**Us) for coeff in H4[i]] for i in Jset(1,4,0)]).transpose()
system_4_1 = B_wa.system_for_constant_solutions([list(row) for row in H4_1_rational_gen])[0]
I_1 = ideal([el for el in system_4_1.minors(1) if el != 0])
PD_I_1 = I_1.primary_decomposition()
for component in PD_I_1:
    component = component.radical()
    if component.dimension() == 0:
        print(component.dimension(), " -> ", [dict_to_point(point,a[2:]) for point in component.variety()], " -> ", component)
    else:
        print(component.dimension(), " |-> ", component)

0  ->  [(0, 0, 0)]  ->  Ideal (a_4, a_3, a_2) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
CPU times: user 42.6 ms, sys: 9.93 ms, total: 52.5 ms
Wall time: 61.5 ms


#### Level variety at level $m=2$

In [45]:
%%time
H4_2_rational_gen = Matrix([[coeff(**Us) for coeff in H4[i]] for i in Jset(2,4,0)]).transpose()
system_4_2 = B_wa.system_for_constant_solutions([list(row) for row in H4_2_rational_gen])[0]
I_2 = ideal([el for el in system_4_2.minors(2) if el != 0])
PD_I_2 = I_2.primary_decomposition()
for component in PD_I_2:
    component = component.radical()
    if component.dimension() == 0:
        print(component.dimension(), " -> ", [dict_to_point(point,a[2:]) for point in component.variety()], " -> ", component)
    else:
        print(component.dimension(), " |-> ", component)

1  |->  Ideal (2*a_2 + a_3, a_3^2 - 24*a_3 - 16*a_4) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(0, 0, 0)]  ->  Ideal (a_4, a_3, a_2) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
CPU times: user 43.5 ms, sys: 20 ms, total: 63.5 ms
Wall time: 69.4 ms


#### Level variety at level $m=3$

In [46]:
%%time
H4_3_rational_gen = Matrix([[coeff(**Us) for coeff in H4[i]] for i in Jset(3,4,0)]).transpose()
system_4_3 = B_wa.system_for_constant_solutions([list(row) for row in H4_3_rational_gen])[0]
I_3 = ideal([el for el in system_4_3.minors(3) if el != 0])
PD_I_3 = I_3.primary_decomposition()
for component in PD_I_3:
    component = component.radical()
    if component.dimension() == 0:
        print(component.dimension(), " -> ", [dict_to_point(point,a[2:]) for point in component.variety()], " -> ", component)
    else:
        print(component.dimension(), " |-> ", component)

0  ->  [(-8, 8, 0)]  ->  Ideal (a_4, a_3 - 8, a_2 + 8) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-8, 24, -24)]  ->  Ideal (a_4 + 24, a_3 - 24, a_2 + 8) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-20, 40, 0)]  ->  Ideal (a_4, a_3 - 40, a_2 + 20) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
1  |->  Ideal (2*a_2 + a_3, a_3^2 - 24*a_3 - 16*a_4) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(0, 0, 0)]  ->  Ideal (a_4, a_3, a_2) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-4, 8, -8)]  ->  Ideal (a_4 + 8, a_3 - 8, a_2 + 4) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
CPU times: user 143 ms, sys: 20.1 ms, total: 163 ms
Wall time: 195 ms


#### Level variety at level $m=5$

In [47]:
%%time
H4_5_rational_gen = Matrix([[coeff(**Us) for coeff in H4[i]] for i in Jset(5,4,0)]).transpose()
system_4_5 = B_wa.system_for_constant_solutions([list(row) for row in H4_5_rational_gen])[0]
I_5 = ideal([el for el in system_4_5.minors(4) if el != 0])
PD_I_5 = I_5.primary_decomposition()
for component in PD_I_5:
    component = component.radical()
    if component.dimension() == 0:
        print(component.dimension(), " -> ", [dict_to_point(point,a[2:]) for point in component.variety()], " -> ", component)
    else:
        print(component.dimension(), " |-> ", component)

0  ->  [(-24, 24, 0)]  ->  Ideal (a_4, a_3 - 24, a_2 + 24) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-24, 72, -72)]  ->  Ideal (a_4 + 72, a_3 - 72, a_2 + 24) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-12, 48, -72)]  ->  Ideal (a_4 + 72, a_3 - 48, a_2 + 12) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-12, 0, 0)]  ->  Ideal (a_4, a_3, a_2 + 12) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-36, 24, 144)]  ->  Ideal (a_4 - 144, a_3 - 24, a_2 + 36) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-36, 120, 0)]  ->  Ideal (a_4, a_3 - 120, a_2 + 36) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-8, 24, -24)]  ->  Ideal (a_4 + 24, a_3 - 24, a_2 + 8) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field
0  ->  [(-8, 8, 0)]  ->  Ideal (a_4, a_3 - 8, a_2 + 8) of Multivariate Polynomial

#### Conclusion

So, at the end, we see that one of the dimension 1 components came from the level variety $\mathcal{X}_2$, and the other 2 dimension 1 components are exactly those that are included in $\mathcal{X}_6$.

It is interesting to remark that these two *new* components in $\mathcal{X}_6 \setminus \mathcal{X}_5$ may contain points from previous level varieties: 

In [48]:
dimension_0 = [component for component in PD_I_6 if component.dimension() == 0]
dimension_1 = [component for component in PD_I_6 if component.dimension() == 1]

In [49]:
point_in_component = []
for point in dimension_0:
    com_in = []
    for i,component in enumerate(dimension_1):
        if all(poly.reduce(point.groebner_basis()) == 0 for poly in component.gens()):
            com_in.append(i)
    point_in_component.append(com_in)
point_in_component

[[], [], [], [], [], [0], [0], [0], [1], [2], [2], [0], [1], [1], [2]]

Finally, we can see that all components of dimension 1 are rational curves that can be parametrized:

In [50]:
dimension_1

[Ideal (4*a_2*a_3 + a_3^2 + 24*a_3 + 16*a_4, a_2^2 + 32*a_2 - 6*a_3 - 4*a_4 + 240) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field,
 Ideal (2*a_2 + a_3, a_3^2 - 152*a_3 - 16*a_4 + 4480) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field,
 Ideal (2*a_2 + a_3, a_3^2 - 24*a_3 - 16*a_4) of Multivariate Polynomial Ring in a_2, a_3, a_4 over Rational Field]

In [51]:
eval_param = lambda par, var : tuple(el(**{f"a_{2+i}":p for (i,p) in enumerate(par)}) for el in var.gens())

In [52]:
R.<t> = QQ[]
param_2 = (-t/2, t, t^2/16 - 3*t/2) # variety from X_2
param_6_1 = (-t/2, t, t^2/16 - 152*t/16 + 4480/16) # variety from X_6

In [53]:
eval_param(param_2, dimension_1[2])

(0, 0)

In [54]:
eval_param(param_6_1, dimension_1[1])

(0, 0)

## Section 8.1 (Hyperbolic for $n=3$)

In this case we explore the operators of the form
$$L = \partial^3 + \frac{a_2}{\cosh(x)^2}\partial + a_3 \frac{ \sinh(x)}{\cosh(x)^3},$$
for two constant values $a_2, a_3$.

For this specific case, we can check that the level varieties satisfy the following properties:
* If $M \equiv 1 \pmod 3$, then $\mathcal{X}_M \setminus \mathcal{X}_{M-2}$ has *only two* new points.
* If $M \equiv 2 \pmod 3$, then $\mathcal{X}_M \setminus \mathcal{X}_{M-2}$ **do not have** any new points.
We can check this fact for $M=4,...,11$ with the following code:

In [55]:
## Setting up the algebraic setting in `dalgebra`.
B = DElliptic(DifferentialRing(QQ), "cosh^2 - cosh_p^2  - 1", "cosh")

DO = DifferentialPolynomialRing(B, "z")
DO_wa = DO.add_constants("a_2", "a_3")
B_wa = DO_wa.base()

cosh = B_wa.gen(); sinh = B_wa.gen_p()
a = [None,None] + [B_wa(f"a_{i}") for i in range(2,4)]
Us = tuple([a[3]*sinh/cosh^3, a[2]/cosh^2])

2025-04-30 12:57:54 INFO     No operation is given: we set a zero derivative.


In [56]:
%%time
X_1 = list(sum([tuple(dict_to_point(point,a[2:]) for point in el[0].full_ideal().variety()) for el in GetEquationsForLevel(3, 1, Us)[-1]], tuple())); X_1

CPU times: user 18.1 ms, sys: 29.9 ms, total: 48.1 ms
Wall time: 54.4 ms


[(0, 0)]

In [57]:
%%time
X_2 = list(sum([tuple(dict_to_point(point,a[2:]) for point in el[0].full_ideal().variety()) for el in GetEquationsForLevel(3, 2, Us)[-1]], tuple())); X_2

CPU times: user 51.9 ms, sys: 10 ms, total: 61.9 ms
Wall time: 60.8 ms


[]

In [58]:
%%time
X_4 = list(sum([tuple(dict_to_point(point,a[2:]) for point in el[0].full_ideal().variety()) for el in GetEquationsForLevel(3, 4, Us)[-1]], tuple())); X_4

CPU times: user 136 ms, sys: 13.8 ms, total: 149 ms
Wall time: 155 ms


[(6, -12), (6, 0)]

In [59]:
%%time
X_5 = list(sum([tuple(dict_to_point(point,a[2:]) for point in el[0].full_ideal().variety()) for el in GetEquationsForLevel(3, 5, Us)[-1]], tuple())); X_5

CPU times: user 214 ms, sys: 9.95 ms, total: 224 ms
Wall time: 224 ms


[]

In [60]:
%%time
X_7 = list(sum([tuple(dict_to_point(point,a[2:]) for point in el[0].full_ideal().variety()) for el in GetEquationsForLevel(3, 7, Us)[-1]], tuple())); X_7

CPU times: user 637 ms, sys: 9.97 ms, total: 646 ms
Wall time: 656 ms


[(18, -48), (18, 12)]

In [61]:
%%time
X_8 = list(sum([tuple(dict_to_point(point,a[2:]) for point in el[0].full_ideal().variety()) for el in GetEquationsForLevel(3, 8, Us)[-1]], tuple())); X_8

CPU times: user 4.09 s, sys: 30 ms, total: 4.12 s
Wall time: 4.11 s


[]

In [62]:
%%time
X_10 = list(sum([tuple(dict_to_point(point,a[2:]) for point in el[0].full_ideal().variety()) for el in GetEquationsForLevel(3, 10, Us)[-1]], tuple())); X_10

CPU times: user 44.9 s, sys: 420 ms, total: 45.3 s
Wall time: 45.3 s


[(36, -120), (36, 48)]

In [ ]:
%%time
X_11 = list(sum([tuple(dict_to_point(point,a[2:]) for point in el[0].full_ideal().variety()) for el in GetEquationsForLevel(3, 11, Us, maple=True)[-1]], tuple())); X_11

As a sumamry, the previous pieces of code shows that the points in the variety $\mathcal{X}_{10}$ are:
* From level $1$: $\{(0, 0)\}$
* From level $4$: $\{(6, -12), (6, 0)\}$
* From level $7$: $\{(18, -48), (18, 12)\}$
* From level $10$: $\{(36, -120), (36, 48)\}$

## Section 8.2 (Elliptic for $n=3$)

In this case we explore the operators of the form
$$L = \partial^3 + a_2\wp\partial + a_3 \wp',$$
for two constant values $a_2, a_3$ and $\wp(x)$ the Weierstrass elliptic function satisfying the differential equations
$$\wp'^2 = \wp^3 + g_2\wp + g_3,$$
for two constants $g_2,g_3$.

In this Section of the paper, we studied the level varieties for $L$ at level $4$. After running algorithm 5 of the paper, we obtained several points. However, we remark here that some on the conditions depend not only on the ansatz variables $a_2,a_3$, but also on the elliptic constants $g_2$ and $g_3$:

In [ ]:
## Setting up the algebraic setting in `dalgebra`.
R.<g_2,g_3> = QQ[]
B = DElliptic(DifferentialRing(R).fraction_field(), "eta_p^2 - eta^3 - g_2*eta - g_3", "eta")

DO = DifferentialPolynomialRing(B, "z")
DO_wa = DO.add_constants("a_2", "a_3")
B_wa = DO_wa.base()

z = DO_wa.gen("z")
eta = B_wa.gen(); eta_p = B_wa.gen_p()
a = [None,None] + [B_wa(f"a_{i}") for i in range(2,4)]
Us = tuple([a[3]*eta_p, a[2]*eta])

In [ ]:
%%time
cases = GetEquationsForLevel(3, 4, Us)[-1]
Is = [el[0].full_ideal() for el in cases]

Here we see three different ideals. The last two ideals show that, no matter the values of $g_2,g_3$, the operators for $a_2 = -3/2$ and $a_3 \in \{0,-3/2\}$ have level exactly $4$. On the other hand, the first ideal show that, when $g_2=0$, we also have the point $(-15/4, -15/8)$ in the level variety.

We now check the structure of the centralizer on these different cases:

#### Generic case 1 ($a_2=-3/2$, $a_3 = -3/2$)

In [ ]:
%%time
Us_g_0 = (-3/2*eta_p, -3/2*eta)
_,CFg_0,_ = GetCentralizer(Us_g_0, 4, starting_level=4, ignore_bound=True)

In [ ]:
[(el[1] if isinstance(el, (list,tuple)) else el).order(z) for el in CFg_0]

In [ ]:
# Operator G_1^*
CFg_0[1]

In [ ]:
# Operator G_2^*
CFg_0[2]

#### Generic case 1 ($a_2=-3/2$, $a_3 = 0$)

In [ ]:
%%time
Us_g_1 = (0, -3/2*eta)
_,CFg_1,_ = GetCentralizer(Us_g_1, 4, starting_level=4, ignore_bound=True)

In [ ]:
[(el[1] if isinstance(el, (list,tuple)) else el).order(z) for el in CFg_1]

In [ ]:
# Operator G_1^*
CFg_1[1]

In [ ]:
# Operator G_2^*
CFg_1[2]

#### Case $g_2=0$ ($a_2=-15/4$, $a_3 = -15/8$)

In [ ]:
%%time
Us_0 = (-15/8*eta_p, -15/4*eta)
_,CF0,_ = GetCentralizer(Us_0, 4, starting_level=4, ignore_bound=True, extra_info=cases[0][0])

In [ ]:
[(el[1] if isinstance(el, (list,tuple)) else el).order(z) for el in CF0]

In [ ]:
# Operator G_1^*
CF0[1]

## Section 8.3 (Rational coefficients)

### Example 8.1 ($n=3$, $m=7$)

In this case we have explored the level variety of rational coefficients of an operator of order $3$ with level $7$. This means we start with an operator of the form

$$L = \partial^3 + \frac{a_2}{x^2}\partial + \frac{a_3}{x^3}$$

Once we get rid of the previous level varieties, we end up with several families of points:

$\begin{array}{|c|c|c|c|}
    \hline
    \text{Family} & \text{\# Cases} & \text{Orders} & \text{Found Relations} \\
    \hline
    \mathcal{F}_1 & 2 & (7, 8) &   \text{None} \\
    \mathcal{F}_2 & 2 & (7, 11) &  \text{None} \\
    \mathcal{F}_3 & 1 & (7, 14) & A_2 - A_1^2 \\
    \hline
\end{array}$

where the points in each family are:

$\begin{array}{rcl}
        \mathcal{F}_1 & = & \left\{(-18, -12), (-18, 48)\right\}\\
        \mathcal{F}_2 & = & \left\{(-30, 0), (-30, 60)\right\}\\
        \mathcal{F}_3 & = & \left\{(-48, 48)\right\}
\end{array}$

In [ ]:
F1 = [(-18, -12), (-18, 48)]
F2 = [(-30, 0), (-30, 60)]
F3 = [(-48,48)]
Fs = [F1, F2, F3]

In [ ]:
B = DifferentialRing(QQ[x], [1]) # ring of polynomials with standard derivation
DO = DifferentialPolynomialRing(B.fraction_field(), "z") # ring of differential operators over Q(x)
B.set_constant(DifferentialRing(QQ))
x = B("x"); z = DO.gen("z")

In [ ]:
example = lambda a_2,a_3: z[3] + a_2/x^2*z[1] + a_3/x^3*z[0]
Us = lambda L : [L.coefficient_full(z[i]) for i in range(2)]

We study now two of the 5 possible cases: the point $(-48,48)$ from the family $\mathcal{F}_3$ and the point $(-18, -12)$ of the family $\mathcal{F}_1$:

#### Case of (-48,48) (family $\mathcal{F}_3$)

In [ ]:
%%time
## Case of (-48,48)
F3_0 = example(*F3[0])
_,CF3_0,_ = GetCentralizer(Us(F3_0), 7, starting_level=7, ignore_bound=True)

In [ ]:
[(el[1] if isinstance(el, (list,tuple)) else el).order(z) for el in CF3_0]

In [ ]:
# G_1^*
CF3_0[1]

#### Case of (-18,-12) (family $\mathcal{F}_1$)

In [ ]:
%%time
F1_0 = example(*F1[0])
_,CF1_0,_ = GetCentralizer(Us(F1_0), 7, starting_level=7, ignore_bound=True)

In [ ]:
[(el[1] if isinstance(el, (list,tuple)) else el).order(z) for el in CF1_0]

In [ ]:
# G_1^*
CF1_0[1]

In [ ]:
# G_2^*
CF1_0[2]

### Example 8.2 ($n=5, m=6$)

In this case we have explored the level variety of rational coefficients of an operator of order $5$ with level $6$. This means we start with an operator of the form

$$L = \partial^5 + \frac{a_2}{x^2}\partial^3 + \frac{a_3}{x^3}\partial^2 + \frac{a_4}{x^4}\partial +  \frac{a_5}{x^5}$$

Once we get rid of the previous level varieties, we end up with several families of points:

$
%\begin{array}{|c|c|c|c|}
%		\hline
%		\text{Family} & \text{\# Cases} & \text{Orders} & \text{Found Relations} \\
%		\hline
%		\mathcal{F}_1 & 6 & (6, 7, 8, 9) &  \text{None}\\
%		\mathcal{F}_2 & 2 & (6, 12, 8, 9) & A_2 - A_1^2 \\
%		\mathcal{F}_3 & 2 & (6, 7, 13, 9) & A_3 - A_1A_2 \\
%		\mathcal{F}_4 & 2 & (6, 7, 13, 14) & A_3 - A_1A_2, A_4 - A_2^2 \\
%		\mathcal{F}_5 & 2 & (6, 12, 8, 14) & A_2 - A_1^2, A_4 - A_1A_3 \\
%		\mathcal{F}_6 & 4 & (6, 12, 13, 9) & A_2 - A_1^2 \\
%		\mathcal{F}_7 & 1 & (6, 7, 8, 14) & A_4 - A_1A_3 \\
%		\mathcal{F}_8 & 3 & (6, 12, 13, 14) & A_2 - A_1^2 \\
%		\mathcal{F}_9 & 2 & (6, 12, 13, 19) & A_2 - A_1^2, A_4 - A_1A_3 \\
%		\mathcal{F}_{10} & 1 & (6, 12, 18, 9) & A_2 - A_1^2, A_3 - A_1^3 \\
%		\mathcal{F}_{11} & 2 & (6, 12, 18, 14) & A_2 - A_1^2, A_3 - A_1^3 \\
%		\mathcal{F}_{12} & 2 & (6, 12, 18, 19) & A_2 - A_1^2, A_3 - A_1^3 \\
%		\mathcal{F}_{13} & 1 & (6, 12, 18, 24) & A_2 - A_1^2, A_3 - A_1^3, A_4 - A_1^4 \\
%		\hline
%\end{array}
\begin{array}{|c|c|c|c|}
                \hline
                \text{Family} & \text{\# Cases} & \text{Orders} & \text{Found Relations} \\
                \hline
                1 & 6 & (6, 7, 8, 9) &  \\
                2 & 2 & (6, 8, 9, 12) & G_4^* - G_1^{ *2} \\
                3 & 2 & (6, 7, 9, 13) & G_4^* - G_1^{ *}G_2^{ *} \\
                4 & 2 & (6, 7, 13, 14) & G_3^* - G_1^{ *}G_2^{ *}, G_4^* - G_2^{ *2} \\
                5 & 2 & (6, 8, 12, 14) & G_3^* - G_1^{ *2}, G_4^* - G_1^{ *}G_2^{ *} \\
                6 & 4 & (6, 9, 12, 13) & G_3^* - G_1^{ *2} \\
                7 & 1 & (6, 7, 8, 14) & G_4^* - G_1^{ *}G_3^{ *} \\
                8 & 3 & (6, 12, 13, 14) & G_2^* - G_1^{ *2} \\
                9 & 2 & (6, 12, 13, 19) & G_2^* - G_1^{ *2}, G_4^* - G_1^{ *}G_3^{ *} \\
                10 & 1 & (6, 9, 12, 18) & G_3^* - G_1^{ *2}, G_4^* - G_1^{ *3} \\
                11 & 2 & (6, 12, 14, 18) & G_2^* - G_1^{ *2}, G_4^* - G_1^{ *3} \\
                12 & 2 & (6, 12, 18, 19) & G_2^* - G_1^{ *2}, G_3^* - G_1^{ *3} \\
                13 & 1 & (6, 12, 18, 24) & G_2^* - G_1^{ *2}, G_3^* - G_1^{ *3}, G_4^* - G_1^{ *4} \\
                \hline
        \end{array}
$

where the points in each family are:

$\begin{array}{rcl}
        \mathcal{F}_1 & = & \left\{(-20, 0, 0, 0), (-25, 55, 70, -70), (-30, 60, 180, 0), (-25, 95, -50, -270), (-30, 120, 0, -720), (-20, 120, -360, 480)\right\}\\
        \mathcal{F}_2 & = & \left\{(-35, 35, 0, 0), (-35, 175, -420, 420)\right\}\\
        \mathcal{F}_3 & = & \left\{(-40, 80, -80, 0), (-40, 160, -320, 320)\right\}\\
        \mathcal{F}_4 & = & \left\{(-60, 240, 0, 0), (-60, 120, 360, -1440)\right\}\\
        \mathcal{F}_5 & = & \left\{(-55, 245, -245, 0), (-55, 85, 235, -640)\right\}\\
        \mathcal{F}_6 & = & \left\{(-50, 40, 240, -240), (-50, 260, -420, 0), (-55, 175, 280, -280), (-55, 155, 340, -1620)\right\}\\
        \mathcal{F}_7 & = & \left\{(-45, 135, -270, 270)\right\}\\
        \mathcal{F}_8 & = & \left\{(-65, 15, 630, 330), (-65, 375, -450, -1470), (-80, 240, 1200, -2880)\right\}\\
        \mathcal{F}_9 & = & \left\{(-100, 200, 880, 0), (-100, 400, 280, -3520)\right\}\\
        \mathcal{F}_{10} & = & \left\{(-85, 255, 130, -770)\right\}\\
        \mathcal{F}_{11} & = & \left\{(-95, 125, 1155, 0), (-95, 445, 195, -3840)\right\}\\
        \mathcal{F}_{12} & = & \left\{(-125, 175, 2730, -2730), (-125, 575, 1530, -7290)\right\}\\
        \mathcal{F}_{13} & = & \left\{(-175, 525, 3955, -8960)\right\}
\end{array}$

In [ ]:
F1 = [(-20, 0, 0, 0), (-25, 55, 70, -70), (-30, 60, 180, 0), (-25, 95, -50, -270), (-30, 120, 0, -720), (-20, 120, -360, 480)]
F2 = [(-35, 35, 0, 0), (-35, 175, -420, 420)]
F3 = [(-40, 80, -80, 0), (-40, 160, -320, 320)]
F4 = [(-60, 240, 0, 0), (-60, 120, 360, -1440)]
F5 = [(-55, 245, -245, 0), (-55, 85, 235, -640)]
F6 = [(-50, 40, 240, -240), (-50, 260, -420, 0), (-55, 175, 280, -280), (-55, 155, 340, -1620)]
F7 = [(-45, 135, -270, 270)]
F8 = [(-65, 15, 630, 330), (-65, 375, -450, -1470), (-80, 240, 1200, -2880)]
F9 = [(-100, 200, 880, 0), (-100, 400, 280, -3520)]
F10 = [(-85, 255, 130, -770)]
F11 = [(-95, 125, 1155, 0), (-95, 445, 195, -3840)]
F12 = [(-125, 175, 2730, -2730), (-125, 575, 1530, -7290)]
F13 = [(-175, 525, 3955, -8960)]
Fs = [F1, F2, F3, F4, F5, F6, F7, F8, F9, F10, F11, F12, F13]

In [ ]:
B = DifferentialRing(QQ[x], [1]) # ring of polynomials with standard derivation
DO = DifferentialPolynomialRing(B.fraction_field(), "z") # ring of differential operators over Q(x)
B.set_constant(DifferentialRing(QQ))
x = B("x"); z = DO.gen("z")

In [ ]:
example = lambda a_2,a_3,a_4,a_5: z[5] + a_2/x^2*z[3] + a_3/x^3*z[2] + a_4/x^4*z[1] + a_5/x^5*z[0]
Us = lambda L : [L.coefficient_full(z[i]) for i in range(4)]

We know showcase the centralizers for the points $(-20,0,0,0)$ from the family $\mathcal{F}_1$ and the point $(-55, 85, 235, -640)$ from the family $\mathcal{F}_5$, checking that the centralizer has the desired structure:

#### Case with point $(-20,0,0,0)$ (family $\mathcal{F}_1$)

In [ ]:
%%time
F1_0 = example(*F1[0])
_,CF1_0,_ = GetCentralizer(Us(F1_0), 6, starting_level=6, ignore_bound=True)

In [ ]:
[(el[1] if isinstance(el, (list,tuple)) else el).order(z) for el in CF1_0]

#### Case with point $(-55, 245, -245, 0)$ (family $\mathcal{F}_5$)

In [ ]:
%%time
F5_1 = example(*F5[1])
_,CF5_1,_ = GetCentralizer(Us(F5_1), 6, starting_level=6, ignore_bound=True)

In [ ]:
[(el[1] if isinstance(el, (list,tuple)) else el).order(z) for el in CF5_1]

In [ ]:
CF5_1[1] # G_1*

In [ ]:
CF5_1[2] # G_2^*

In [ ]:
CF5_1[3][1] # G_3^*

In [ ]:
CF5_1[1].sym_power(2, z)

In [ ]:
CF5_1[3][1] == CF5_1[1].sym_power(2, z) # G_3^* == G_1^*^2

##### Extra from paper
###### Computing more relations between generators

In [ ]:
from dalgebra.commutators.commutator import reduce_as_module, BC_ideal

In [ ]:
result = []
for i,family in enumerate(Fs):
    res_family = []
    for point in family:
        print("#####################################################")
        print(f"++ Starting computation of case in family {i+1}: {point}")
        L = example(*point)
        U = Us(L)
        print(f"++     Computing centralizer of the operator...")
        _,C,_ = GetCentralizer(U, 6, starting_level=6, ignore_bound=True)
        print(f"++     Computing the B.C. ideal for the operator...")
        I = BC_ideal(L, tuple(C), z)
        print(f"++     The ideal {'is' if I.is_prime() else 'is not'} prime")
        res_family.append((C,I,I.is_prime()))
    result.append(res_family)